# Gabarito - Exercícios da Aula 02

**Módulo 02 - NumPy e Pandas Avançado para Bioinformática Forense**

---

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

## Exercício 1: Pipeline de QC de Amostras

### Enunciado
Usando o dataset `df_forense`:
a) Filtre amostras com qualidade "Alta" E não degradadas
b) Agrupe por analista e calcule estatísticas completas
c) Identifique o analista com melhor performance (maior média, menor CV)
d) Crie um pivot table cruzando tipo × prioridade com contagens

In [ ]:
# Recriar dataset forense
n_amostras = 100
df_forense = pd.DataFrame({
    'id': [f'AM{i:03d}' for i in range(1, n_amostras + 1)],
    'tipo': np.random.choice(['sangue', 'saliva', 'tecido', 'osso'], n_amostras,
                            p=[0.35, 0.25, 0.25, 0.15]),
    'dna_conc': np.round(np.random.uniform(0.5, 80, n_amostras), 2),
    'ratio_260_280': np.round(np.random.uniform(1.2, 2.1, n_amostras), 2),
    'degradado': np.random.choice([True, False], n_amostras, p=[0.2, 0.8]),
    'analista': np.random.choice(['Silva', 'Santos', 'Oliveira', 'Costa'], n_amostras),
    'prioridade': np.random.choice(['alta', 'media', 'baixa'], n_amostras,
                                   p=[0.2, 0.5, 0.3])
})

df_forense['qualidade'] = pd.cut(
    df_forense['ratio_260_280'],
    bins=[0, 1.6, 1.8, 3.0],
    labels=['Baixa', 'Media', 'Alta']
)

In [ ]:
# a) Filtrar amostras com qualidade Alta E não degradadas
amostras_excelentes = df_forense[
    (df_forense['qualidade'] == 'Alta') &
    (df_forense['degradado'] == False)
]

print(f'=== a) Amostras Excelentes (Alta qualidade + Não degradadas) ===')
print(f'Total: {len(amostras_excelentes)} amostras ({len(amostras_excelentes)/len(df_forense)*100:.1f}% do total)')
display(amostras_excelentes.head(10))

In [ ]:
# b) Agrupar por analista com estatísticas completas
def coeficiente_variacao(x):
    return (x.std() / x.mean()) * 100 if x.mean() != 0 else np.nan

stats_analista = df_forense.groupby('analista')['dna_conc'].agg(
    n='count',
    media='mean',
    mediana='median',
    dp='std',
    cv_percentual=coeficiente_variacao,
    min='min',
    max='max'
).round(2)

print('=== b) Estatísticas Completas por Analista ===')
display(stats_analista)

In [ ]:
# c) Identificar melhor analista
stats_norm = stats_analista.copy()
stats_norm['score_media'] = (stats_norm['media'] - stats_norm['media'].min()) / (stats_norm['media'].max() - stats_norm['media'].min())
stats_norm['score_cv'] = 1 - (stats_norm['cv_percentual'] - stats_norm['cv_percentual'].min()) / (stats_norm['cv_percentual'].max() - stats_norm['cv_percentual'].min())
stats_norm['score_composto'] = (stats_norm['score_media'] + stats_norm['score_cv']) / 2

melhor_analista = stats_norm['score_composto'].idxmax()

print('=== c) Ranking de Performance dos Analistas ===')
display(stats_norm[['media', 'cv_percentual', 'score_composto']].sort_values('score_composto', ascending=False).round(3))
print(f'\n★ Melhor analista (score composto): {melhor_analista}')

In [ ]:
# d) Pivot table: tipo × prioridade
pivot_tipo_prioridade = pd.pivot_table(
    df_forense,
    values='id',
    index='tipo',
    columns='prioridade',
    aggfunc='count',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print('=== d) Contagem de Amostras: Tipo × Prioridade ===')
display(pivot_tipo_prioridade)

### Explicação
- **Filtragem booleana combinada** (`&`) é essencial para triagem rigorosa de amostras forenses
- **Score composto** permite ranking multidimensional; em laboratórios reais, KPIs de analistas são monitorados continuamente
- **Pivot tables com margins** fornecem totais automáticos para relatórios QC
- O **CV (Coeficiente de Variação)** mede consistência; valores <30% são geralmente aceitáveis em extrações de DNA

## Exercício 2: Integração de Dados Multi-fonte

### Enunciado
Crie dois DataFrames:
- `perfis_str`: 10 amostras com genótipos de 5 marcadores
- `metadados_lab`: informações de coleta e processamento

Realize merge e responda:
a) Quantas amostras têm dados completos?
b) Quais amostras estão faltando em cada fonte?
c) Calcule frequência alélica para um marcador após o merge

In [ ]:
# Criar perfis STR (10 amostras)
perfis_str = pd.DataFrame({
    'amostra_id': [f'AM{i:03d}' for i in range(1, 11)],
    'D3S1358': ['15/17', '14/16', '15/15', '16/18', '14/15', '15/17', '14/18', '16/16', '15/16', '17/18'],
    'TH01': ['7/9.3', '6/9', '8/8', '7/7', '9/9.3', '7/9', '6/9.3', '8/9', '7/8', '9/9'],
    'D21S11': ['28/30', '29/31', '28/28', '30/31', '28/29', '29/30', '28/31', '30/30', '29/29', '28/30'],
    'D18S51': ['14/17', '13/16', '15/17', '14/15', '16/17', '14/16', '15/15', '13/17', '14/14', '16/18'],
    'vWA': ['14/18', '15/17', '16/18', '14/16', '17/18', '15/18', '14/17', '16/16', '15/15', '17/17']
})

# Criar metadados (8 amostras, algumas diferentes)
metadados_lab = pd.DataFrame({
    'amostra_id': [f'AM{i:03d}' for i in [1, 2, 3, 5, 7, 8, 10, 11]],
    'tipo': ['sangue', 'saliva', 'tecido', 'sangue', 'osso', 'saliva', 'sangue', 'tecido'],
    'data_coleta': pd.date_range('2026-10-01', periods=8, freq='2D'),
    'caso': ['CASO_001', 'CASO_001', 'CASO_002', 'CASO_003', 'CASO_004', 'CASO_004', 'CASO_005', 'CASO_005'],
    'dna_conc_ng_ul': [25.4, 18.7, 42.1, 8.3, 3.2, 31.5, 15.9, 22.8]
})

print('=== Perfis STR (10 amostras) ===')
display(perfis_str)
print('\n=== Metadados Lab (8 amostras) ===')
display(metadados_lab)

In [ ]:
# Merge com indicador de origem
merged = pd.merge(perfis_str, metadados_lab, on='amostra_id', how='outer', indicator=True)

# a) Amostras com dados completos
completas = merged[merged['_merge'] == 'both']
print(f'=== a) Amostras com Dados Completos ===')
print(f'Total: {len(completas)} amostras')
print(f'IDs: {sorted(completas["amostra_id"].tolist())}')

# b) Amostras faltantes em cada fonte
so_perfil = merged[merged['_merge'] == 'left_only']
so_metadados = merged[merged['_merge'] == 'right_only']

print(f'\n=== b) Amostras Faltantes ===')
print(f'Com perfil mas SEM metadados: {sorted(so_perfil["amostra_id"].tolist())}')
print(f'Com metadados mas SEM perfil: {sorted(so_metadados["amostra_id"].tolist())}')

In [ ]:
# c) Frequência alélica do marcador TH01 após merge
th01_genotipos = completas['TH01'].dropna()

alelos_th01 = []
for genotipo in th01_genotipos:
    a1, a2 = genotipo.split('/')
    alelos_th01.extend([a1, a2])

freq_th01 = pd.Series(alelos_th01).value_counts(normalize=True).sort_index()

print('=== c) Frequências Alélicas TH01 (amostras completas) ===')
for alelo, freq in freq_th01.items():
    print(f'  Alelo {alelo}: {freq:.4f} ({freq*100:.1f}%)')
print(f'\nTotal de alelos contados: {len(alelos_th01)} ({len(alelos_th01)//2} indivíduos × 2)')

### Explicação
- **Merge outer com indicator** revela discrepâncias entre fontes de dados — comum em laboratórios onde perfis e metadados são gerados em sistemas separados
- Amostras **sem metadados** violam cadeia de custódia e devem ser investigadas
- Amostras **sem perfil** podem indicar falha na extração/amplificação
- Cálculo de frequências usa apenas **dados completos** para evitar viés

## Exercício 3: Análise de Matriz de Distâncias Genéticas

### Enunciado
Dada uma matriz Fst de 5 populações:
a) Calcule autovalores e determine quantas dimensões explicam >80% da variância
b) Normalize a matriz (Z-score por linha)
c) Identifique o par de populações mais diferenciado
d) Visualize como heatmap

In [ ]:
# Matriz Fst de 5 populações (valores simulados realistas)
populacoes = ['Europeu', 'Africano', 'Asiático', 'Americano', 'Oceânico']
fst_matrix = np.array([
    [0.000, 0.120, 0.095, 0.045, 0.110],  # Europeu
    [0.120, 0.000, 0.180, 0.135, 0.165],  # Africano
    [0.095, 0.180, 0.000, 0.110, 0.145],  # Asiático
    [0.045, 0.135, 0.110, 0.000, 0.125],  # Americano
    [0.110, 0.165, 0.145, 0.125, 0.000]   # Oceânico
])

print('Matriz Fst:')
fst_df = pd.DataFrame(fst_matrix, index=populacoes, columns=populacoes)
display(fst_df)

In [ ]:
# a) Autovalores e variância explicada
autovalores, autovetores = np.linalg.eig(fst_matrix)

# Usar valores reais (matriz simétrica → autovalores reais)
autovalores_real = np.real(autovalores)
var_explicada = autovalores_real / np.sum(autovalores_real)
var_acumulada = np.cumsum(var_explicada)

print('=== a) Análise de Componentes Principais ===')
n_dims_80 = 0
for i, (eig, var, acum) in enumerate(zip(autovalores_real, var_explicada, var_acumulada)):
    marker = ''
    if acum >= 0.8 and (i == 0 or var_acumulada[i-1] < 0.8):
        marker = ' ← >80% atingido aqui'
        n_dims_80 = i + 1
    print(f'  PC{i+1}: autovalor={eig:.4f}, var={var*100:.1f}%, acumulada={acum*100:.1f}%{marker}')

print(f'\nDimensões necessárias para >80% da variância: {n_dims_80}')

In [ ]:
# b) Normalização Z-score por linha
medias_linha = np.mean(fst_matrix, axis=1, keepdims=True)
dp_linha = np.std(fst_matrix, axis=1, keepdims=True)
fst_normalizado = (fst_matrix - medias_linha) / dp_linha

print('=== b) Matriz Fst Normalizada (Z-score por população) ===')
fst_norm_df = pd.DataFrame(np.round(fst_normalizado, 3), index=populacoes, columns=populacoes)
display(fst_norm_df)
print('\nInterpretação: valores positivos = acima da média de diferenciação daquela população')
print('             valores negativos = abaixo da média (mais similar às outras)')

In [ ]:
# c) Par de populações mais diferenciado
# Excluir diagonal (distância de cada população consigo mesma = 0)
fst_no_diag = fst_matrix.copy()
np.fill_diagonal(fst_no_diag, -np.inf)

idx_max = np.unravel_index(np.argmax(fst_no_diag), fst_no_diag.shape)
pop1 = populacoes[idx_max[0]]
pop2 = populacoes[idx_max[1]]
fst_max = fst_matrix[idx_max]

print('=== c) Par Mais Diferenciado ===')
print(f'  População 1: {pop1}')
print(f'  População 2: {pop2}')
print(f'  Fst máximo: {fst_max:.4f}')
print(f'\nInterpretação forense: Este par requer marcadores específicos para discriminação confiável')

In [ ]:
# d) Heatmap da matriz Fst
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(
    fst_df,
    annot=True,
    fmt='.3f',
    cmap='YlOrRd',
    vmin=0,
    vmax=0.2,
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Fst'}
)
plt.title('Matriz de Distância Genética (Fst) entre Populações', fontsize=14, pad=20)
plt.xlabel('População', fontsize=12)
plt.ylabel('População', fontsize=12)
plt.tight_layout()
plt.show()

print('Nota: Cores mais escuras indicam maior diferenciação genética')

### Explicação
- **Autovalores** indicam a importância de cada componente principal na explicação da variabilidade genética
- **Normalização Z-score** permite comparar padrões relativos de diferenciação independente da escala absoluta
- O **par mais diferenciado** (geralmente Africano vs Asiático ou similar) define os limites de resolução do painel de marcadores
- Em forense, matrizes Fst são usadas para **correção de subestrutura populacional** (theta/Fst correction) nos cálculos de LR
- **Heatmaps** facilitam a comunicação de resultados complexos em relatórios periciais

---

## Conceitos-Chave Revisados

| Conceito | Aplicação Forense |
|----------|-------------------|
| Vetorização NumPy | Processamento rápido de RFUs e genótipos |
| Broadcasting | Normalização de matrizes de alelos |
| groupby + agg customizado | Relatórios QC por analista/lote |
| merge com indicator | Auditoria de integridade de dados |
| pivot_table | Relatórios tabulares para laudos |
| Álgebra linear (eig) | PCA de estrutura populacional |
| Heatmap Seaborn | Visualização de matrizes de distância |

---

> ⚖️ **Nota sobre Validação Forense**: Todo pipeline computacional em laboratório forense deve ser validado segundo ISO 17025 e diretrizes SWGDAM antes de uso em casos reais. Documente versões (`np.__version__`, `pd.__version__`) e parâmetros utilizados.